# SE374 · Bone suppression → VinDr-CXR detection (Kaggle / Colab)

Notebook dùng chung cho cả nhóm: **mỗi lần chạy = 1 thí nghiệm** (`A_original`, `B1_rajaraman`, `B2_hmchuong`).
Mọi bước nằm trong `scripts/run_experiment.py`, nên chạy trên Kaggle, Colab hay máy cá nhân đều ra cùng một quy trình.

**Quy tắc:** tất cả thí nghiệm YOLO phải chạy trên **cùng một nền tảng** (nhóm chọn Kaggle T4) với **cùng `YOLO_ARGS`**.

### Kaggle (khuyên dùng)
1. *Add Input* → competition **VinBigData Chest X-ray Abnormalities Detection** (phải bấm *Join* cuộc thi trước) và dataset **xhlulu/vinbigdata-chest-xray-resized-png-1024x1024**.
2. *Settings* → Accelerator **GPU T4**, **Internet: On** (cần xác minh số điện thoại).
3. Sửa ô cấu hình, rồi **Save Version → Save & Run All (Commit)**: chạy nền tối đa ~12 giờ, tắt trình duyệt cũng được.
4. Xong: mở tab *Output*, tải `outputs/<EXPERIMENT>.zip`.

### Colab
Chạy từng ô. `runs/` được lưu vào Google Drive để train tiếp được khi bị ngắt (`RESUME = True`).
Dữ liệu phải tải lại mỗi phiên bằng Kaggle API (cần `kaggle.json`).

In [ ]:
# ===== CẤU HÌNH =====
EXPERIMENT = "A_original"          # A_original | B1_rajaraman | B2_hmchuong
REPO = "https://github.com/HinnoNguyen/SE374.git"
BRANCH = "main"

# GIỐNG NHAU cho mọi thí nghiệm. Không đổi sau khi đã chạy thí nghiệm đầu tiên.
YOLO_ARGS = "--model yolo11s.pt --imgsz 1024 --epochs 50 --batch 8 --workers 4"

# Train tiếp một lần chạy bị ngắt:
RESUME = False
# Kaggle: last.pt nằm trong Output của version trước -> thêm version đó làm Input rồi điền đường dẫn, ví dụ
# "/kaggle/input/se374-vindr/SE374/runs/vindr/A_original/weights/last.pt". Colab: để trống (đã nằm trên Drive).
RESUME_FROM = ""

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

ON_KAGGLE = Path("/kaggle/input").exists()
ON_COLAB = "google.colab" in sys.modules
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("/content") if ON_COLAB else Path.cwd()
REPO_DIR = WORK / "SE374"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)

if ON_COLAB:  # keep runs/ on Drive so an interrupted training can be resumed
    from google.colab import drive
    drive.mount("/content/drive")
    drive_runs = Path("/content/drive/MyDrive/SE374_work/runs")
    drive_runs.mkdir(parents=True, exist_ok=True)
    if not Path("runs").exists():
        os.symlink(drive_runs, "runs")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "ensemble-boxes",
                "pytorch-msssim", "pydicom", "h5py"], check=True)
import torch
print("platform:", "Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "local")
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

### (Chỉ Colab) Tải dữ liệu
Kaggle không cần ô này, vì dữ liệu đã được gắn sẵn làm Input.
Kích thước gốc của ảnh lấy từ file meta đi kèm bộ PNG, hoặc từ `data/vindr_train_sizes.csv` trong repo (tạo 1 lần trên Kaggle rồi commit).

In [ ]:
if ON_COLAB:
    from google.colab import files
    if not Path("~/.kaggle/kaggle.json").expanduser().exists():
        print("Upload kaggle.json"); up = files.upload()
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        shutil.move("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    data = Path("/content/data"); data.mkdir(exist_ok=True)
    if not (data / "train.csv").exists():
        subprocess.run(["kaggle", "competitions", "download", "-c", "vinbigdata-chest-xray-abnormalities-detection",
                        "-f", "train.csv", "-p", str(data)], check=True)
        for z in data.glob("*.zip"):
            shutil.unpack_archive(z, data); z.unlink()
    if not any(data.rglob("*.png")):
        subprocess.run(["kaggle", "datasets", "download", "-d", "xhlulu/vinbigdata-chest-xray-resized-png-1024x1024",
                        "-p", str(data), "--unzip"], check=True)

In [ ]:
INPUT = "/kaggle/input" if ON_KAGGLE else "/content/data" if ON_COLAB else "data/raw"

if RESUME and RESUME_FROM:  # bring last.pt of the previous session back to the same place
    dst = Path("runs/vindr") / EXPERIMENT / "weights" / "last.pt"
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(RESUME_FROM, dst)

cmd = [sys.executable, "scripts/run_experiment.py", "--experiment", EXPERIMENT, "--auto-find", INPUT, *YOLO_ARGS.split()]
if RESUME:
    cmd.append("--resume")
print(" ".join(cmd))
subprocess.run(cmd, check=True)

### Lưu kết quả
File zip chứa: `results/` (CSV, biểu đồ), `data/vindr_splits.csv`, `data/vindr_train_sizes.csv` và trọng số `best.pt` / `last.pt`.

Sau khi tải về máy:
- Giải nén vào repo. **Commit** `results/`, `data/vindr_splits.csv`, `data/vindr_train_sizes.csv` (nhỏ, cần để mọi người dùng chung một cách chia tập).
- **Không commit** `.pt`. Đưa `best.pt` lên **GitHub Releases** (hoặc Drive) rồi ghi link vào README.
- Ảnh VinDr (kể cả ảnh đã xóa xương) **không bao giờ** đưa lên GitHub (giấy phép dữ liệu).

In [ ]:
import zipfile
out = WORK / "outputs"; out.mkdir(exist_ok=True)
zpath = out / f"{EXPERIMENT}.zip"
items = [Path("results"), Path("data/vindr_splits.csv"), Path("data/vindr_train_sizes.csv"),
         Path("runs/vindr") / EXPERIMENT / "weights" / "best.pt", Path("runs/vindr") / EXPERIMENT / "weights" / "last.pt"]
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for item in items:
        for f in ([item] if item.is_file() else sorted(item.rglob("*")) if item.exists() else []):
            if f.is_file():
                z.write(f, f.as_posix())
print("saved", zpath, f"{zpath.stat().st_size / 1e6:.1f} MB")
if ON_COLAB:
    shutil.copy2(zpath, "/content/drive/MyDrive/SE374_work/")
    print("copied to Google Drive / SE374_work")